### Exemplo de Chatbot com Transformers

Este exemplo demonstra como criar um chatbot simples usando um modelo Transformer pré-treinado da biblioteca `transformers` da Hugging Face. Usaremos o modelo `microsoft/DialoGPT-medium`, projetado para IA conversacional.

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Carregar modelo pré-treinado e tokenizador
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")


Agora, vamos criar um loop de bate-papo simples. Você pode digitar suas mensagens e o chatbot responderá. Para sair do bate-papo, digite `quit` ou `exit`.

In [ ]:
def run_chatbot():
    chat_history_ids = None
    print("\n--- Chatbot Started ---")
    print("Type 'quit' or 'exit' to end the conversation.")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["quit", "exit"]:
            print("Chatbot: Goodbye!")
            break

        # Codifique a entrada do usuário e gere a máscara de atenção
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
        print(f"\n[PASSO 1] Input IDs (Tokens): {new_user_input_ids.tolist()}")

        # Adicione a nova entrada do usuário ao histórico do chat
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids
        # Criar máscara de atenção
        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        # Gere uma resposta com a máscara de atenção fornecida
        print("[PASSO 2] O modelo está calculando a probabilidade do próximo token...")
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            num_return_sequences=1
        )

        # Decodifique
        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
        print(f"Chatbot: {response}")

In [ ]:
run_chatbot()